# Feature Engineering - ShipmentSure

This notebook creates new features and transforms existing ones to improve prediction.

**Steps covered:**
1. Label encoding for categorical columns
2. Create new derived features
3. Binning (discretization) of numeric features
4. Feature importance using correlation
5. Save the engineered dataset

---

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import os

sns.set_theme(style="whitegrid")
print("Libraries imported!")

In [ ]:
# Load the dataset
df = pd.read_csv('Train.csv')

# Drop ID column (not useful for prediction)
if 'ID' in df.columns:
    df = df.drop('ID', axis=1)

# Rename target column
df = df.rename(columns={'Reached.on.Time_Y.N': 'On_Time_Delivery'})

print(f"Dataset shape: {df.shape}")
df.head()

## 1. Label Encoding for Categorical Columns

Machine learning models need numeric data. We convert categorical text into numbers using Label Encoding.

In [ ]:
# Identify categorical columns
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f"Categorical columns: {cat_cols}")

# Show unique values before encoding
for col in cat_cols:
    print(f"\n{col}: {df[col].unique()}")

In [ ]:
# Apply Label Encoding
le = LabelEncoder()
encoding_map = {}  # Store the mappings for reference

for col in cat_cols:
    df[col + '_encoded'] = le.fit_transform(df[col])
    mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    encoding_map[col] = mapping
    print(f"{col} encoding: {mapping}")

print("\n✅ Label encoding complete!")
df.head()

## 2. Create New Derived Features

New features can capture patterns that individual features miss.

In [ ]:
# Feature 1: Cost per gram
# This tells us how expensive the product is relative to its weight
df['Cost_per_gram'] = df['Cost_of_the_Product'] / df['Weight_in_gms']
print("✅ Created 'Cost_per_gram' = Cost / Weight")

# Feature 2: Discount percentage relative to cost
# How big is the discount compared to the product cost?
df['Discount_pct'] = (df['Discount_offered'] / df['Cost_of_the_Product']) * 100
print("✅ Created 'Discount_pct' = (Discount / Cost) * 100")

# Feature 3: High-value product flag
# Products costing more than the median are considered high-value
cost_median = df['Cost_of_the_Product'].median()
df['High_value_product'] = (df['Cost_of_the_Product'] > cost_median).astype(int)
print(f"✅ Created 'High_value_product' (1 if cost > {cost_median})")

# Feature 4: Heavy shipment flag
weight_median = df['Weight_in_gms'].median()
df['Heavy_shipment'] = (df['Weight_in_gms'] > weight_median).astype(int)
print(f"✅ Created 'Heavy_shipment' (1 if weight > {weight_median})")

# Feature 5: Frequent buyer flag
purchases_median = df['Prior_purchases'].median()
df['Frequent_buyer'] = (df['Prior_purchases'] > purchases_median).astype(int)
print(f"✅ Created 'Frequent_buyer' (1 if prior_purchases > {purchases_median})")

print(f"\nNew columns added: {df.shape[1] - 11} new features")
df[['Cost_per_gram', 'Discount_pct', 'High_value_product', 'Heavy_shipment', 'Frequent_buyer']].head()

## 3. Binning (Discretization)

Convert continuous values into categories (bins) to simplify patterns.

In [ ]:
# Bin the Weight column into categories
df['Weight_category'] = pd.cut(df['Weight_in_gms'], 
                                bins=[0, 2000, 4000, 6000],
                                labels=['Light', 'Medium', 'Heavy'])
print("Weight categories:")
print(df['Weight_category'].value_counts())

# Bin the Discount column
df['Discount_level'] = pd.cut(df['Discount_offered'],
                               bins=[0, 10, 30, 50, 65],
                               labels=['Low', 'Medium', 'High', 'Very High'])
print("\nDiscount levels:")
print(df['Discount_level'].value_counts())

In [ ]:
# Visualize delivery rates by the new bins
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Weight category vs delivery
weight_delivery = df.groupby('Weight_category')['On_Time_Delivery'].mean() * 100
weight_delivery.plot(kind='bar', ax=axes[0], color=['#3498db', '#2ecc71', '#e74c3c'])
axes[0].set_title('On-Time Delivery % by Weight Category', fontweight='bold')
axes[0].set_ylabel('On-Time Delivery %')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

# Discount level vs delivery
discount_delivery = df.groupby('Discount_level')['On_Time_Delivery'].mean() * 100
discount_delivery.plot(kind='bar', ax=axes[1], color=['#3498db', '#2ecc71', '#f39c12', '#e74c3c'])
axes[1].set_title('On-Time Delivery % by Discount Level', fontweight='bold')
axes[1].set_ylabel('On-Time Delivery %')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

## 4. Feature Importance (Correlation with Target)

In [ ]:
# Select only numeric columns for correlation
numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()

# Calculate correlation with target
correlations = df[numeric_features].corr()['On_Time_Delivery'].drop('On_Time_Delivery')
correlations = correlations.abs().sort_values(ascending=False)

print("Feature Importance (Absolute Correlation with Target):")
print("="*50)
for feat, corr in correlations.items():
    bar = '█' * int(corr * 50)
    print(f"{feat:30s} | {corr:.4f} | {bar}")

In [ ]:
# Visualize feature importance
plt.figure(figsize=(10, 6))
colors = ['#e74c3c' if c > 0.1 else '#3498db' for c in correlations.values]
correlations.plot(kind='barh', color=colors)
plt.title('Feature Importance (|Correlation| with On-Time Delivery)', fontweight='bold')
plt.xlabel('Absolute Correlation')
plt.axvline(x=0.1, color='red', linestyle='--', alpha=0.5, label='Threshold (0.1)')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Save Engineered Dataset

In [ ]:
# Save the final engineered dataset
os.makedirs('data', exist_ok=True)

# Drop the original categorical columns (keep encoded versions)
df_final = df.drop(columns=cat_cols + ['Weight_category', 'Discount_level'])

df_final.to_csv('data/engineered_data.csv', index=False)
print(f"✅ Engineered dataset saved to 'data/engineered_data.csv'")
print(f"   Shape: {df_final.shape}")
print(f"   Columns: {df_final.columns.tolist()}")

## Summary

| Technique | Features Created |
|-----------|------------------|
| Label Encoding | Warehouse_block_encoded, Mode_of_Shipment_encoded, Product_importance_encoded, Gender_encoded |
| Derived Features | Cost_per_gram, Discount_pct, High_value_product, Heavy_shipment, Frequent_buyer |
| Binning | Weight_category, Discount_level |

**Key finding:** Discount_offered and Weight_in_gms tend to have the strongest relationship with delivery timeliness.